In [11]:
import numpy as np
import pandas as pd
import yfinance as yf
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots  

In [12]:


# Single currency parameter (example: "AUDNZD=X", "EURUSD=X", "GBPUSD=X")
currency = "CHFJPY=X"  # Replace with your desired currency pair

# Fetch full historical data
df_max = yf.download(currency, period="max", interval="1d")

# Clean multi-index columns if returned by yfinance
if isinstance(df_max.columns, pd.MultiIndex):
    df_max.columns = df_max.columns.get_level_values(0)

df_max = df_max.dropna(subset=["Close", "High", "Low"])

# Filter 1-Year and 10-Year subsets
latest_date = df_max.index.max()
df_1y = df_max.loc[df_max.index >= (latest_date - pd.DateOffset(years=1))].copy()
df_10y = df_max.loc[df_max.index >= (latest_date - pd.DateOffset(years=10))].copy()

# Reusable charting & reporting function
def display_horizon_analysis(df_slice, horizon_label, color_code):
    # Quantitative calculations
    df_slice["Log_Return"] = np.log(df_slice["Close"] / df_slice["Close"].shift(1))
    ann_return = df_slice["Log_Return"].mean() * 252
    ann_volatility = df_slice["Log_Return"].std() * np.sqrt(252)
    mean_price = float(df_slice["Close"].mean())
    ath_price = float(df_slice["High"].max())
    ath_date = df_slice["High"].idxmax()
    atl_price = float(df_slice["Low"].min())
    atl_date = df_slice["Low"].idxmin()

    # Interactive Plotly chart
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=df_slice.index,
            y=df_slice["Close"],
            mode="lines",
            name=f"{currency} Close",
            line=dict(color=color_code, width=1.6),
            hovertemplate="<b>Date</b>: %{x|%Y-%m-%d}<br><b>Price</b>: %{y:.4f}<extra></extra>",
        )
    )
    fig.add_hline(
        y=mean_price,
        line=dict(color="#FFB300", width=1.0, dash="dash"),
        annotation_text=f"Mean ({mean_price:.4f})",
        annotation_position="bottom right",
        annotation_font=dict(color="#FFB300", size=10),
    )
    fig.update_layout(
        title=dict(text=f"<b>{currency}</b> — {horizon_label} Horizon", font=dict(size=15, color="#FFFFFF")),
        height=380,
        template="plotly_dark",
        plot_bgcolor="#131722",
        paper_bgcolor="#131722",
        hovermode="x unified",
        margin=dict(l=40, r=40, t=50, b=30),
    )
    fig.show()

    # Performance report output
    print(f"=== {horizon_label.upper()} PERFORMANCE REPORT ({currency}) ===")
    print(f"Annualized Return     : {ann_return:.2%}")
    print(f"Annualized Volatility : {ann_volatility:.2%}")
    print(f"Mean Price Level      : {mean_price:.4f}")
    print(f"Period High (ATH)     : {ath_price:.4f} on {ath_date.strftime('%Y-%m-%d')}")
    print(f"Period Low  (ATL)     : {atl_price:.4f} on {atl_date.strftime('%Y-%m-%d')}")
    print("-" * 55)


[*********************100%***********************]  1 of 1 completed


In [13]:
display_horizon_analysis(df_1y, "1-Year", "#00D4FF")


=== 1-YEAR PERFORMANCE REPORT (CHFJPY=X) ===
Annualized Return     : 0.74%
Annualized Volatility : 6.90%
Mean Price Level      : 197.7305
Period High (ATH)     : 204.4123 on 2026-04-21
Period Low  (ATL)     : 187.6000 on 2026-09-30
-------------------------------------------------------


In [14]:
display_horizon_analysis(df_10y, "10-Year", "#4D56FF")


=== 10-YEAR PERFORMANCE REPORT (CHFJPY=X) ===
Annualized Return     : 5.84%
Annualized Volatility : 7.41%
Mean Price Level      : 139.7685
Period High (ATH)     : 204.4123 on 2026-04-21
Period Low  (ATL)     : 96.5100 on 2019-01-01
-------------------------------------------------------


In [15]:
display_horizon_analysis(df_max, "All-Time", "#CB00E6")


=== ALL-TIME PERFORMANCE REPORT (CHFJPY=X) ===
Annualized Return     : 3.49%
Annualized Volatility : 11.02%
Mean Price Level      : 115.7111
Period High (ATH)     : 204.4123 on 2026-04-21
Period Low  (ATL)     : 75.0800 on 2008-12-05
-------------------------------------------------------
